In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_cfb_2024_rushing():
    url = "https://www.sports-reference.com/cfb/years/2024-rushing.html"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Find the table using the correct id
    table = soup.find("table", id="rushing_standard")
    if not table:
        raise ValueError("Rushing stats table with id='rushing_standard' not found.")

    # Get column headers from second row in thead
    header_rows = table.find("thead").find_all("tr")
    columns = [th.get_text(strip=True) for th in header_rows[1].find_all("th")]
    columns = columns[1:]  # Drop the "Rk" column which is <th>, not <td>

    # Parse each row of player data
    data = []
    for row in table.find("tbody").find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue  # skip repeated headers in tbody
        cells = [td.get_text(strip=True) for td in row.find_all("td")]
        if not cells:
            continue
        data.append(cells)

    # Create dataframe
    df = pd.DataFrame(data, columns=columns)

    # Optional: convert numeric-looking columns
    for col in df.columns:
        df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

    return df

if __name__ == "__main__":
    df = scrape_cfb_2024_rushing()
    print(df.head())
    df.to_csv("cfb_2024_rushing_standard.csv", index=False)


AttributeError: 'DataFrame' object has no attribute 'str'

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_cfb_2024_rushing():
    url = "https://www.sports-reference.com/cfb/years/2024-rushing.html"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Locate the table with the correct id
    table = soup.find("table", id="rushing_standard")
    if not table:
        raise ValueError("Could not find the rushing stats table with id='rushing_standard'.")

    # Extract column names from the second row in thead
    header_rows = table.find("thead").find_all("tr")
    columns = [th.get_text(strip=True) for th in header_rows[1].find_all("th")]
    columns = columns[1:]  # Remove 'Rk' column from headers

    # Extract player rows from tbody
    data = []
    for row in table.find("tbody").find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue  # Skip repeated header rows
        cells = [td.get_text(strip=True) for td in row.find_all("td")]
        if not cells:
            continue
        data.append(cells)

    # Create the DataFrame
    df = pd.DataFrame(data, columns=columns)

    # Safely convert numeric-looking columns
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

    return df

if __name__ == "__main__":
    df = scrape_cfb_2024_rushing()
    print(df.head())
    df.to_csv("cfb_2024_rushing_standard.csv", index=False)


AttributeError: 'DataFrame' object has no attribute 'dtype'

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_cfb_2024_rushing():
    url = "https://www.sports-reference.com/cfb/years/2024-rushing.html"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", id="rushing_standard")
    if not table:
        raise ValueError("Could not find table with id='rushing_standard'.")

    # Step 1: Extract headers from the first row with <td> inside <tbody>
    tbody = table.find("tbody")
    first_valid_row = next(row for row in tbody.find_all("tr") if row.find_all("td"))
    columns = [td['data-stat'] for td in first_valid_row.find_all("td")]

    # Step 2: Extract all player data rows
    data = []
    for row in tbody.find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue  # skip internal headers
        tds = row.find_all("td")
        if not tds:
            continue
        values = [td.get_text(strip=True) for td in tds]
        data.append(values)

    # Step 3: Create DataFrame
    df = pd.DataFrame(data, columns=columns)

    # Step 4: Convert numeric-looking columns
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

    return df

if __name__ == "__main__":
    df = scrape_cfb_2024_rushing()
    print(df.head())
    df.to_csv("cfb_2024_rushing_only_tbody.csv", index=False)


       name_display      team_name_abbr conf_abbr  games  rush_att  rush_yds  \
0     Ashton Jeanty         Boise State       MWC   14.0     374.0    2601.0   
1  Cameron Skattebo       Arizona State    Big 12   13.0     293.0    1711.0   
2  Omarion Hampton*      North Carolina       ACC   12.0     281.0    1660.0   
3     Bryson Daily*                Army  American   13.0     310.0    1659.0   
4      Tre Stewart*  Jacksonville State      CUSA   14.0     278.0    1638.0   

   rush_yds_per_att  rush_td  rush_yds_per_g   rec  rec_yds  rec_yds_per_rec  \
0               7.0     29.0           185.8  23.0    138.0              6.0   
1               5.8     21.0           131.6  45.0    605.0             13.4   
2               5.9     15.0           138.3  38.0    373.0              9.8   
3               5.4     32.0           127.6   0.0      0.0              NaN   
4               5.9     25.0           117.0  18.0    234.0             13.0   

   rec_td  rec_yds_per_g  scrim_att  y

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_cfb_2024_passing():
    url = "https://www.sports-reference.com/cfb/years/2024-passing.html"
    response = requests.get(url)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Locate table with ID 'passing_standard'
    table = soup.find("table", id="passing_standard")
    if not table:
        raise ValueError("Could not find table with id='passing_standard'.")

    # Extract column headers from the first data row
    tbody = table.find("tbody")
    first_valid_row = next(row for row in tbody.find_all("tr") if row.find_all("td"))
    columns = [td['data-stat'] for td in first_valid_row.find_all("td")]

    # Extract data rows
    data = []
    for row in tbody.find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue  # skip sub-header rows
        tds = row.find_all("td")
        if not tds:
            continue
        values = [td.get_text(strip=True) for td in tds]
        data.append(values)

    # Create DataFrame
    df = pd.DataFrame(data, columns=columns)

    # Convert number-looking strings
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].str.replace(',', ''), errors='ignore')

    return df

if __name__ == "__main__":
    df = scrape_cfb_2024_passing()
    print(df.head())
    df.to_csv("cfb_2024_passing_standard.csv", index=False)


         name_display team_name_abbr conf_abbr  games  pass_cmp  pass_att  \
0        Kyle McCord*       Syracuse       ACC   13.0     391.0     592.0   
1       Cameron Ward*     Miami (FL)       ACC   13.0     305.0     454.0   
2        Jaxson Dart*       Ole Miss       SEC   13.0     276.0     398.0   
3    Shedeur Sanders*       Colorado    Big 12   13.0     353.0     477.0   
4  Garrett Nussmeier*            LSU       SEC   13.0     337.0     525.0   

   pass_cmp_pct  pass_yds  pass_td  pass_td_pct  pass_int  pass_int_pct  \
0          66.0    4779.0     34.0          5.7      12.0           2.0   
1          67.2    4313.0     39.0          8.6       7.0           1.5   
2          69.3    4279.0     29.0          7.3       6.0           1.5   
3          74.0    4134.0     37.0          7.8      10.0           2.1   
4          64.2    4052.0     29.0          5.5      12.0           2.3   

   pass_yds_per_att  pass_adj_yds_per_att  pass_yds_per_cmp  pass_yds_per_g  \
0      